In [1]:
def hex2dec(v):
    if '0' <= v <= '9':
        return ord(v) - ord('0')
    else:
        return ord(v.lower()) - ord('a') + 10

In [2]:
def hexa2dec(val):
    res = 0
    for c in val:
        res = res * 16 + hex2dec(c)
    return res

In [3]:
def decodeTxt(txt):
    res = ""
    for i in range(0, len(txt), 2):
        code = hexa2dec(txt[i:i + 2])
        res += chr(code)
    return res

In [34]:


def lettreCesar(let, d):
    if let == ' ':
        return ' '
    return chr((ord(let) - ord('A') + d) % 26 + ord('A'))

In [35]:
def cesar(txt, d):
    return "".join(lettreCesar(c, d) for c in txt)

In [4]:
def decodeCesar(txt, mot):
    return [d for d in range(26)
            if mot in cesar(txt, d)]

# Usage : on lit le fichier texte et on cherche un mot connu
with open("DM4-image.txt") as f:  contenu = f.read().upper()
decalages = decodeCesar(contenu, "JAMES")
for d in decalages:
    print(f"d={d} : {cesar(contenu, d)}")

NameError: name 'cesar' is not defined

In [37]:
def bit(nb, n):
    return (nb >> n) & 1

In [38]:
def selectImage(mat, f=lambda x: x):
    return [[f(val) for val in ligne] for ligne in mat]

In [5]:
def sautImage(mat, s):
    return mat[::s]

M=[[21, 122, 223 ], [ 243, 144, 45 ], [ 24, 125, 226 ],[ 246, 147, 48 ], [ 27, 128, 229 ] ]

sautImage(M, 2)



[[21, 122, 223], [24, 125, 226], [27, 128, 229]]

In [40]:
def recomposeImage(mat, r):
    res = []
    for ligne in mat:
        nouvelle_ligne = []
        for debut in range(r):
            bits = ligne[debut::r]          # indices debut, debut+r, debut+2r…
            # Compléter à 8 bits par des 0 à droite
            bits = bits + [0] * (8 - len(bits))
            valeur = 0
            for b in bits:
                valeur = (valeur << 1) | b
            nouvelle_ligne.append(valeur)
        res.append(nouvelle_ligne)
    return res

recomposeImage([[1,1,0,1,0,0,0,0,1]], 3)

[[192, 128, 32]]

In [41]:
from PIL import Image
import numpy as np

# 1. Charger l'image en niveaux de gris
img = Image.open("DM4-image.bmp").convert("L")
mat = [list(row) for row in np.array(img)]

# 2. Extraire les bits de poids faible (LSB steganography)
bits_mat = selectImage(mat, lambda x: bit(x, 0))

# 3. Sous-échantillonner si nécessaire et reconstruire
r = 8   # paramètre à ajuster selon la structure de l'image
decoded = recomposeImage(bits_mat, r)

# 4. Sauvegarder
out = Image.fromarray(np.array(decoded, dtype=np.uint8))
out.save("message_cache.bmp")
print("Image décodée sauvegardée.")



Image décodée sauvegardée.


In [42]:
from PIL import Image
import numpy as np

img = Image.open("DM4-image.bmp").convert("L")
mat = np.array(img)

# 1. Extraire TOUS les LSB en une seule séquence plate
bits = []
for row in mat:
    for pixel in row:
        bits.append(pixel & 1)  # bit de poids faible

# 2. Regrouper les bits par 8 → octets → caractères
message = ""
for i in range(0, len(bits) - 7, 8):
    octet = 0
    for j in range(8):
        octet = (octet << 1) | bits[i + j]
    if octet == 0:          # octet nul = fin du message
        break
    if 32 <= octet <= 126:  # caractère ASCII imprimable
        message += chr(octet)
    else:
        message += f"[{octet}]"

print("Message caché :", message)

Message caché : 


In [6]:
# ============================================================
#  DM4 Informatique – James Donb
#  Solution complète – sans import, sans int() pour Q1-Q2
# ============================================================

# -----------------------------------------------------------
# Q1 : hex2dec – un seul caractère hexadécimal -> entier
#      INTERDIT : int()
# -----------------------------------------------------------
def hex2dec(v: str) -> int:
    """
    Convertit un caractère hexadécimal (0-9 ou a-f/A-F) en entier.
    Exemple : hex2dec("a") -> 10, hex2dec("7") -> 7
    """
    digits = "0123456789abcdef"
    return digits.index(v.lower())


# -----------------------------------------------------------
# Q2 : hexa2dec – chaîne hexadécimale -> entier
#      INTERDIT : int()
# -----------------------------------------------------------
def hexa2dec(val: str) -> int:
    """
    Convertit une chaîne hexadécimale en entier.
    Exemple : hexa2dec("7a") -> 122
    """
    result = 0
    for c in val:
        result = result * 16 + hex2dec(c)
    return result


# -----------------------------------------------------------
# Q3 : decodeTxt – suite de codes hex -> chaîne lisible
# -----------------------------------------------------------
def decodeTxt(txt: str) -> str:
    """
    Convertit une suite de codes hexadécimaux (par paires) en chaîne ASCII.
    Exemple : decodeTxt("6d707369") -> "mpsi"
    """
    result = ""
    for i in range(0, len(txt), 2):
        code = hexa2dec(txt[i:i+2])
        result += chr(code)
    return result


# -----------------------------------------------------------
# Q4 : lettreCesar – décale une lettre majuscule de d
# -----------------------------------------------------------
def lettreCesar(let: str, d: int) -> str:
    """
    Décale la lettre majuscule `let` de `d` positions (modulo 26).
    Renvoie un espace si `let` est un espace.
    Exemple : lettreCesar('A', 5) -> 'F'
    """
    if let == ' ':
        return ' '
    return chr((ord(let) - ord('A') + d) % 26 + ord('A'))


# -----------------------------------------------------------
# Q5 : cesar – chiffre un texte (majuscules + espaces) par décalage d
# -----------------------------------------------------------
def cesar(txt: str, d: int) -> str:
    """
    Applique le chiffrement de César avec décalage d à tout le texte.
    Exemple : cesar("ABU VW", 5) -> "FGZ AB"
    """
    return "".join(lettreCesar(c, d) for c in txt)


# -----------------------------------------------------------
# Q6 : decodeCesar – renvoie tous les décalages pour lesquels
#      txt contient mot
# -----------------------------------------------------------
def decodeCesar(txt: str, mot: str) -> list:
    """
    Renvoie la liste de tous les décalages d ∈ {0,...,25} pour lesquels
    le texte déchiffré avec décalage d contient le mot `mot`.
    """
    decalages = []
    for d in range(26):
        if mot in cesar(txt, d):
            decalages.append(d)
    return decalages


# -----------------------------------------------------------
# Q7 : bit – renvoie le bit de poids 2^n d'un entier nb
# -----------------------------------------------------------
def bit(nb: int, n: int) -> int:
    """
    Renvoie le bit de poids 2^n du nombre nb.
    Exemple : bit(201, 3) -> 1  (201 = 11001001₂, bit 3 = 1)
              bit(201, 5) -> 0
    """
    return (nb >> n) & 1


# -----------------------------------------------------------
# Q8 : selectImage – applique f à chaque élément de la matrice
# -----------------------------------------------------------
def selectImage(mat: list, f=lambda x: x) -> list:
    """
    Renvoie une nouvelle matrice de même taille dont chaque élément
    est l'image par f de l'élément correspondant de mat.
    Exemple : selectImage(M, lambda x: x%2)
    """
    return [[f(v) for v in row] for row in mat]


# -----------------------------------------------------------
# Q9 : sautImage – sélectionne les lignes d'indice 0, s, 2s, 3s, …
# -----------------------------------------------------------
def sautImage(mat: list, s: int) -> list:
    """
    Renvoie une sous-matrice composée des lignes d'indices 0, s, 2s, …
    Exemple : sautImage(M, 2) garde les lignes 0, 2, 4, ...
    """
    return [mat[i] for i in range(0, len(mat), s)]


# -----------------------------------------------------------
# Q10 : recomposeImage – regroupe les bits par colonnes entrelacées
#       en octets (complétés à droite par des 0)
# -----------------------------------------------------------
def recomposeImage(mat: list, r: int) -> list:
    """
    Pour chaque ligne de mat (composée de bits 0/1) :
      - Crée r groupes : groupe j contient les éléments aux indices
        j, j+r, j+2r, … (sélection par pas de r)
      - Complète chaque groupe à 8 bits (pad de 0 à droite)
      - Convertit chaque groupe en un octet (bit de poids fort en premier)
    Renvoie la matrice des octets ainsi formés (r colonnes par ligne).

    Exemple :
      recomposeImage([[1,1,0,1,0,0,0,0,1]], 3)
      -> [[192, 128, 32]]
         car 192 = 11000000₂  (indices 0,3,6 → [1,1,0] padded)
             128 = 10000000₂  (indices 1,4,7 → [1,0,0] padded)
              32 = 00100000₂  (indices 2,5,8 → [0,0,1] padded)
    """
    result = []
    for row in mat:
        n = len(row)
        new_row = []
        for j in range(r):
            # Sélection par pas de r, au plus 8 éléments
            bits = []
            k = j
            while k < n and len(bits) < 8:
                bits.append(row[k])
                k += r
            # Complétion à 8 bits
            bits8 = bits + [0] * (8 - len(bits))
            # Conversion binaire -> entier (bit 0 = poids fort)
            val = sum(bits8[i] * (2 ** (7 - i)) for i in range(8))
            new_row.append(val)
        result.append(new_row)
    return result


# -----------------------------------------------------------
# Fonctions utilitaires pour lire/écrire un BMP 8bpp
# (niveaux de gris, sans module)
# -----------------------------------------------------------
def lire_bmp(chemin: str) -> tuple:
    """
    Lit un fichier BMP 8bpp et renvoie (matrice_pixels, largeur, hauteur).
    La matrice est indexée mat[ligne][colonne], ligne 0 = haut de l'image.
    """
    with open(chemin, "rb") as f:
        data = f.read()

    def read_int(d, start, size):
        val = 0
        for i in range(size):
            val += d[start + i] * (256 ** i)
        return val

    offset = read_int(data, 10, 4)
    largeur = read_int(data, 18, 4)
    hauteur = read_int(data, 22, 4)
    bpp     = read_int(data, 28, 2)

    if bpp != 8:
        raise ValueError(f"BMP {bpp}bpp non supporté, seul 8bpp est géré ici.")

    row_size = (largeur + 3) // 4 * 4   # multiple de 4

    # BMP stocke les lignes de bas en haut
    mat = []
    for row in range(hauteur - 1, -1, -1):
        debut = offset + row * row_size
        mat.append(list(data[debut:debut + largeur]))

    return mat, largeur, hauteur


def ecrire_bmp(chemin: str, mat: list, largeur: int, hauteur: int) -> None:
    """
    Écrit une matrice de niveaux de gris (0-255) dans un fichier BMP 8bpp.
    """
    row_size = (largeur + 3) // 4 * 4
    pixel_data_size = row_size * hauteur
    file_size = 1078 + pixel_data_size   # 14 + 40 + 256*4 = 1078

    def to_le4(n):
        return bytes([n & 0xFF, (n >> 8) & 0xFF,
                      (n >> 16) & 0xFF, (n >> 24) & 0xFF])

    def to_le2(n):
        return bytes([n & 0xFF, (n >> 8) & 0xFF])

    file_header = (b'BM' + to_le4(file_size)
                   + b'\x00\x00\x00\x00' + to_le4(1078))
    dib_header  = (to_le4(40) + to_le4(largeur) + to_le4(hauteur)
                   + to_le2(1) + to_le2(8)
                   + to_le4(0) + to_le4(pixel_data_size)
                   + to_le4(2835) + to_le4(2835)
                   + to_le4(256) + to_le4(256))
    palette     = b''.join(bytes([i, i, i, 0]) for i in range(256))

    # Pixels : BMP stocke de bas en haut
    pixel_bytes = bytearray()
    for row_idx in range(hauteur - 1, -1, -1):
        ligne = mat[row_idx][:largeur]
        pixel_bytes.extend(ligne)
        pixel_bytes.extend(b'\x00' * (row_size - largeur))

    with open(chemin, 'wb') as f:
        f.write(file_header + dib_header + palette + bytes(pixel_bytes))


# -----------------------------------------------------------
# Q11 : Application du pipeline à l'image, sauvegarde du résultat
# -----------------------------------------------------------
def decoder_image(chemin_entree: str, chemin_sortie: str,
                  r_recompose: int = 160) -> None:
    """
    Pipeline complet :
      1. Lecture du BMP 8bpp
      2. selectImage(mat, lambda x: x%2)  → bits LSB
      3. recomposeImage(bits, r_recompose) → image décodée
      4. Sauvegarde en BMP

    Avec r=160 et une image 1280×853 :
      chaque groupe de 8 bits espacés de 160 forme un octet
      → image de sortie de largeur 160, même hauteur 853.
    """
    mat, w, h = lire_bmp(chemin_entree)
    print(f"Image lue : {w}×{h}")

    # Étape 1 : extraire les bits LSB
    bits = selectImage(mat, lambda x: x % 2)

    # Étape 2 : recombiner par groupes entrelacés
    decoded = recomposeImage(bits, r_recompose)

    new_w = len(decoded[0]) if decoded else 0
    new_h = len(decoded)
    print(f"Image décodée : {new_w}×{new_h}")

    ecrire_bmp(chemin_sortie, decoded, new_w, new_h)
    print(f"Image sauvegardée dans : {chemin_sortie}")


# -----------------------------------------------------------
# Tests rapides (à retirer avant rendu si nécessaire)
# -----------------------------------------------------------
if __name__ == "__main__":

    # --- Q1 ---
    assert hex2dec("7") == 7
    assert hex2dec("a") == 10
    assert hex2dec("F") == 15
    print("Q1 OK")

    # --- Q2 ---
    assert hexa2dec("7a") == 122
    assert hexa2dec("ff") == 255
    assert hexa2dec("00") == 0
    print("Q2 OK")

    # --- Q3 ---
    assert decodeTxt("6d707369") == "mpsi"
    print("Q3 OK")

    # --- Q4 ---
    assert lettreCesar('A', 5)  == 'F'
    assert lettreCesar('V', 5)  == 'A'
    assert lettreCesar(' ', 5)  == ' '
    print("Q4 OK")

    # --- Q5 ---
    assert cesar("ABU VW", 5) == "FGZ AB"
    print("Q5 OK")

    # --- Q7 ---
    assert bit(201, 5) == 0
    assert bit(201, 3) == 1
    assert bit(201, 0) == 1
    assert bit(201, 7) == 1
    print("Q7 OK")

    # --- Q8 ---
    M = [[21, 123, 12], [43, 44, 181]]
    assert selectImage(M, lambda x: x % 2) == [[1, 1, 0], [1, 0, 1]]
    print("Q8 OK")

    # --- Q9 ---
    M2 = [[i] for i in range(5)]   # 5 lignes
    assert sautImage(M2, 2) == [[0], [2], [4]]
    print("Q9 OK")

    # --- Q10 ---
    assert recomposeImage([[1, 1, 0, 1, 0, 0, 0, 0, 1]], 3) == [[192, 128, 32]]
    print("Q10 OK")

    # --- Q11 : décodage de l'image ---
    # Adapter les chemins selon votre machine
    import os
    bmp_entree = "DM4-image.bmp"
    bmp_sortie = "DM4-image-decodee.bmp"
    if os.path.exists(bmp_entree):
        decoder_image(bmp_entree, bmp_sortie, r_recompose=160)
        print(f"Message caché sauvegardé dans {bmp_sortie}")
    else:
        print(f"(fichier {bmp_entree} introuvable, passé)")


Q1 OK
Q2 OK
Q3 OK
Q4 OK
Q5 OK
Q7 OK
Q8 OK
Q9 OK
Q10 OK
Image lue : 1280×853
Image décodée : 160×853
Image sauvegardée dans : DM4-image-decodee.bmp
Message caché sauvegardé dans DM4-image-decodee.bmp


In [4]:
# 1. Vous exécutez d'abord le décodage pour obtenir la matrice 'decoded'
mat, w, h = lire_bmp("DM4-image.bmp")
bits = selectImage(mat, lambda x: x % 2)
decoded = recomposeImage(bits, 160)

# 2. On extrait les caractères ASCII correspondants à la première ligne d'octets
phrase_mystere = "".join(chr(octet) for octet in decoded[0])
print("Phrase brute extraite :", phrase_mystere)

# 3. Si la phrase semble incompréhensible, elle est chiffrée par César.
# On utilise la fonction Q6 en cherchant un mot probable (ex: "JAMES", "DONB", "INFORMATIQUE", "CODE")
# Ou on affiche simplement les 26 possibilités :
for d in range(26):
    print(f"Décalage {d} : {cesar(phrase_mystere, d)}")

Phrase brute extraite :                                                                                                                                                                 
Décalage 0 : NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
Décalage 1 : OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
Décalage 2 : PPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPPP
Décalage 3 : QQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ
Décalage 4 : RRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR